In [ ]:
# ==============================================================================
# PROJECT: SI (Synthetic Intelligence Architecture)
# SUB-SYSTEM: Cross-Tokenizer Alignment via Vectorized Mapping & Subtoken Sync
# VERSION: v9.2 (Production Stable - Logits Dimension Fixed)
# INFRASTRUCTURE: PyTorch, Transformers, BitsAndBytes
# PLATFORM: Google Colab - T4 GPU (16GB VRAM)
# ==============================================================================

# 1. Global kütüphanelerin kurulumu
!pip install -q transformers torch huggingface_hub bitsandbytes accelerate

import os
import time
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import notebook_login

print("==================================================================")
print("🧠 SI (SYNTHETIC INTELLIGENCE) ENGINE V9.2 - LOGITS DIMENSION FIXED")
print("==================================================================")

print("\n📢 IMPORTANT NOTE FOR USERS:")
print("• This notebook uses Meta's Llama-3.2-3B-Instruct which is a GATED model.")
print("• Ensure your Hugging Face account has requested and received permission from Meta.")
print("• Provide an Access Token with 'Read' permissions in the widget below.\n")

print("🔒 [SI SECURITY] Requesting Hugging Face Authentication:")
notebook_login()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n🚀 [SI ENGINE] Active Hardware Layer: {device.upper()}")

quant_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model_expert_name = "meta-llama/Llama-3.2-3B-Instruct"
model_amateur_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("\n⏳ Downloading and loading neural weights into VRAM...")
tokenizer_expert = AutoTokenizer.from_pretrained(model_expert_name)
tokenizer_amateur = AutoTokenizer.from_pretrained(model_amateur_name)

model_expert = AutoModelForCausalLM.from_pretrained(model_expert_name, quantization_config=quant_config_4bit)
model_amateur = AutoModelForCausalLM.from_pretrained(model_amateur_name, quantization_config=quant_config_4bit)

# ==============================================================================
# VOCABULARY MAPPING (CRITICAL FIX: MODEL CONFIG VOCAB SIZE USED)
# ==============================================================================
print("\n🔄 Building 1-to-1 Vocabulary Mapping Matrix...")
# Modelin gerçek çıktı katmanı boyutu referans alınıyor (Örn: 128256)
expert_vocab_size = model_expert.config.vocab_size
expert_to_amateur_map = torch.full((expert_vocab_size,), -1, dtype=torch.long, device=device)

for token_str, expert_id in tokenizer_expert.get_vocab().items():
    if expert_id < expert_vocab_size:
        amateur_ids = tokenizer_amateur.encode(token_str, add_special_tokens=False)
        if len(amateur_ids) == 1:
            # v7-v8 arası sinsi Python liste atama bug'ı temizlendi, net skaler atandı
            expert_to_amateur_map[expert_id] = amateur_ids[0]

print("🎯 Vocabulary Mapping Completed Successfully!")

# ==============================================================================
# HIZ ÖLÇÜMÜ İÇİN MODEL WARM-UP (DRY-RUN)
# ==============================================================================
print("\n🔥 Warming up CUDA kernels (Pre-compiling for accurate benchmarks)...")
dummy_input = torch.tensor([[tokenizer_expert.eos_token_id]], device=device)
_ = model_expert(dummy_input, use_cache=False)
_ = model_amateur(torch.tensor([[tokenizer_amateur.eos_token_id]], device=device), use_cache=False)
print("==================================================================")
print("💬 SI INTERACTIVE PRODUCTION CONSOLE")
print("Tip: Adjust alpha and penalty values dynamically per prompt type.")
print("==================================================================")

while True:
    user_query = input("\n👤 User Prompt: ")

    if user_query.lower() in ['exit', 'quit']:
        print("\n🤖 SI Engine: Session safely terminated. Goodbye!")
        break

    if not user_query.strip():
        continue

    try:
        print("\n🎛️ Tune Hyperparameters (Press Enter to use default settings):")
        alpha_input = input("  • Contrastive Alpha (Default 0.40): ").strip()
        alpha = float(alpha_input) if alpha_input else 0.40

        rep_input = input("  • Repetition Penalty (Default 0.30): ").strip()
        repetition_penalty = float(rep_input) if rep_input else 0.30

        window_input = input("  • Penalty Window Size (Default 20): ").strip()
        penalty_window = int(window_input) if window_input else 20
    except ValueError:
        print("⚠️ Invalid inputs detected. Falling back to default parameters.")
        alpha = 0.40
        repetition_penalty = 0.30
        penalty_window = 20

    msg_expert = tokenizer_expert.apply_chat_template([{"role": "user", "content": user_query}], tokenize=False, add_generation_prompt=True)
    msg_amateur = tokenizer_amateur.apply_chat_template([{"role": "user", "content": user_query}], tokenize=False, add_generation_prompt=True)

    input_ids_expert = tokenizer_expert([msg_expert], return_tensors="pt").input_ids.to(device)
    input_ids_amateur = tokenizer_amateur([msg_amateur], return_tensors="pt").input_ids.to(device)

    print("\n🤖 SI Output: ", end="", flush=True)

    soft_limit = 100
    absolute_max = 180

    past_expert, past_amateur = None, None
    mask_expert = torch.ones_like(input_ids_expert).to(device)
    mask_amateur = torch.ones_like(input_ids_amateur).to(device)

    curr_expert = input_ids_expert
    curr_amateur = input_ids_amateur

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    total_tokens_generated = 0
    generated_token_ids = []

    for token_index in range(absolute_max):
        with torch.no_grad():
            out_expert = model_expert(input_ids=curr_expert, past_key_values=past_expert, attention_mask=mask_expert, use_cache=True)
            logits_expert = out_expert.logits[:, -1, :]
            past_expert = out_expert.past_key_values

            out_amateur = model_amateur(input_ids=curr_amateur, past_key_values=past_amateur, attention_mask=mask_amateur, use_cache=True)
            logits_amateur = out_amateur.logits[:, -1, :]
            past_amateur = out_amateur.past_key_values

            log_probs_expert = F.log_softmax(logits_expert, dim=-1)
            log_probs_amateur = F.log_softmax(logits_amateur, dim=-1)

            mapped_amateur_log_probs = torch.zeros_like(log_probs_expert)
            valid_mask = expert_to_amateur_map >= 0
            mapped_amateur_log_probs[0, valid_mask] = log_probs_amateur[0, expert_to_amateur_map[valid_mask]]

            fused_log_probs = log_probs_expert - (alpha * mapped_amateur_log_probs)

            # Pencereli Repetition Penalty Katmanı
            if generated_token_ids:
                recent_ids = set(generated_token_ids[-penalty_window:])
                for tid in recent_ids:
                    fused_log_probs[0, tid] -= repetition_penalty

            if token_index >= (absolute_max - 5):
                fused_log_probs[0, tokenizer_expert.eos_token_id] += 100.0

            next_token = torch.argmax(fused_log_probs, dim=-1, keepdim=True)
            total_tokens_generated += 1
            generated_token_ids.append(next_token.item())

            chosen_word = tokenizer_expert.decode([next_token.item()])
            print(chosen_word, end="", flush=True)

            if token_index >= soft_limit and any(p in chosen_word for p in [".", "!", "?"]):
                break

            if next_token.item() == tokenizer_expert.eos_token_id:
                break

            curr_expert = next_token
            mask_expert = torch.cat([mask_expert, torch.ones((1, 1), device=device)], dim=-1)

            # === BATCH SUBTOKEN SYNC ENGINE ===
            amateur_token_id = expert_to_amateur_map[next_token.item()].item()
            if amateur_token_id >= 0:
                curr_amateur = torch.tensor([[amateur_token_id]], device=device)
                mask_amateur = torch.cat([mask_amateur, torch.ones((1, 1), device=device)], dim=-1)
            else:
                amateur_sub_tokens = tokenizer_amateur.encode(chosen_word, add_special_tokens=False)
                if amateur_sub_tokens:
                    if len(amateur_sub_tokens) > 1:
                        batch_input = torch.tensor([amateur_sub_tokens[:-1]], device=device)
                        batch_mask = torch.cat([mask_amateur, torch.ones((1, batch_input.shape[-1]), device=device)], dim=-1)
                        out_amateur = model_amateur(input_ids=batch_input, past_key_values=past_amateur, attention_mask=batch_mask, use_cache=True)
                        past_amateur = out_amateur.past_key_values
                        mask_amateur = batch_mask

                    curr_amateur = torch.tensor([[amateur_sub_tokens[-1]]], device=device)
                    mask_amateur = torch.cat([mask_amateur, torch.ones((1, 1), device=device)], dim=-1)
                else:
                    curr_amateur = torch.tensor([[tokenizer_amateur.eos_token_id]], device=device)
                    mask_amateur = torch.cat([mask_amateur, torch.ones((1, 1), device=device)], dim=-1)

    end_time = time.time()
    peak_vram = torch.cuda.max_memory_allocated() / 1e9
    duration = end_time - start_time
    tokens_per_sec = total_tokens_generated / duration if duration > 0 else 0

    print("\n" + "-" * 60)
    print(f"📊 VERIFIED SI BENCHMARK METRICS (V9.2):")
    print(f"  • Config Used      : alpha={alpha}, rep_penalty={repetition_penalty}, window={penalty_window}")
    print(f"  • Generation Speed : {tokens_per_sec:.2f} tokens/second")
    print(f"  • Peak VRAM Usage  : {peak_vram:.2f} GB")
    print("-" * 60)


🧠 SI (SYNTHETIC INTELLIGENCE) ENGINE V9.2 - LOGITS DIMENSION FIXED

📢 IMPORTANT NOTE FOR USERS:
• This notebook uses Meta's Llama-3.2-3B-Instruct which is a GATED model.
• Ensure your Hugging Face account has requested and received permission from Meta.
• Provide an Access Token with 'Read' permissions in the widget below.

🔒 [SI SECURITY] Requesting Hugging Face Authentication:

🚀 [SI ENGINE] Active Hardware Layer: CUDA

⏳ Downloading and loading neural weights into VRAM...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


🔄 Building 1-to-1 Vocabulary Mapping Matrix...
🎯 Vocabulary Mapping Completed Successfully!

🔥 Warming up CUDA kernels (Pre-compiling for accurate benchmarks)...
💬 SI INTERACTIVE PRODUCTION CONSOLE
Tip: Adjust alpha and penalty values dynamically per prompt type.

👤 User Prompt: Who won the Nobel Prize in Physics in 1823?

🎛️ Tune Hyperparameters (Press Enter to use default settings):
  • Contrastive Alpha (Default 0.40): 
  • Repetition Penalty (Default 0.30): 
  • Penalty Window Size (Default 20): 

🤖 SI Output: 

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


There was no 1823 Nobel Prize in Physics. The first Nobel Prizes were awarded in 1901, 50 years after 1823.<|eot_id|>
------------------------------------------------------------
📊 VERIFIED SI BENCHMARK METRICS (V9.2):
  • Config Used      : alpha=0.4, rep_penalty=0.3, window=20
  • Generation Speed : 7.02 tokens/second
  • Peak VRAM Usage  : 7.27 GB
------------------------------------------------------------

👤 User Prompt: What was the first domestic automobile brand produced in Türkiye in the 1970s, and what were its technical specifications?

🎛️ Tune Hyperparameters (Press Enter to use default settings):
  • Contrastive Alpha (Default 0.40): 
  • Repetition Penalty (Default 0.30): 
  • Penalty Window Size (Default 20): 

🤖 SI Output: I  was  unable to verify the 1st domestic automobile brand, produced in 1970s, in Türkiye.<|eot_id|>
------------------------------------------------------------
📊 VERIFIED SI BENCHMARK METRICS (V9.2):
  • Config Used      : alpha=0.4, rep_penalty=0.3